In [1]:
%reset -f

In [2]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [3]:
ol = Overlay('resizer-zcu102.bit')

In [ ]:
help(ol)

In [ ]:
# help(img2axis.register_map)

In [4]:
def img_to_axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip.register_map.data_port = buffer.physical_address

    # Set end_of_stream 
    ip.register_map.end_of_stream = eos

    # Set frame_no to 88
    ip.register_map.frame_cnt = frame_cnt
    # Start the IP core by setting the ap_start bit in CTRL register
    ip.register_map.CTRL.AP_START=1
    
def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (b << 16) | (g << 8) | r  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

def unpack_frame(packed):
    H, W_packed,ch = packed.shape  # (480, 160,4)
    W = W_packed * ch            # Unpacked width = 640
    output_tensor = np.reshape(frame, (H, W, 1))
    return output_tensor


In [5]:
import socket
import numpy as np
import time  # for simulating delay between frames

def send_frame(unpacked_frame, DEST_IP = '192.168.100.119',DEST_PORT = 5005):

    
    # Remove the singleton channel dimension (shape becomes 480x640)
    frame_bytes = unpacked_frame.squeeze(axis=2).tobytes()

    # Create a TCP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect((DEST_IP, DEST_PORT))

    # Optionally, send the frame size first (so the receiver knows what to expect)
    frame_size = len(frame_bytes)
    sock.sendall(frame_size.to_bytes(4, byteorder='big'))  # Send 4-byte length

    # Send the frame
    sock.sendall(frame_bytes)

    # Close the connection
    sock.close()

    print("Frame sent successfully.")

In [6]:
from pynq.lib.video import *
vdma = ol.axi_vdma_0

vdma.readchannel.reset()
vdma.readchannel.mode = VideoMode(width=160, height=480, bits_per_pixel=32)

vdma.readchannel.start()



In [7]:
print(f"VDMA.running={vdma.readchannel.running},\nVDMA.activeframe={vdma.readchannel.activeframe},\nVDMA.mode={vdma.readchannel.mode}")

VDMA.running=True,
VDMA.activeframe=0,
VDMA.mode=VideoMode: width=160 height=480 bpp=32 fps=60


In [8]:
img_fname='1920x1080-full-hd-nature-landscape.jpg'
#img_fname='PM5644-1920x1080.gif'

In [9]:
buff_o=image_to_RGB(img_fname)

Packed buffer shape: (1080, 1920), dtype: uint32


In [10]:
img_to_axis(ol.img2axis_0,buff_o,True,4)

In [11]:
frame = vdma.readchannel.readframe()

In [12]:
print(f"type(frame)={type(frame)},frame.shape={frame.shape},frame.dtype={frame.dtype}")

type(frame)=<class 'pynq.buffer.PynqBuffer'>,frame.shape=(480, 160, 4),frame.dtype=uint8


In [13]:
unpacked_frame = unpack_frame(frame)
print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape},\nunpacked_frame.dtype={unpacked_frame.dtype}")

type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),
unpacked_frame.dtype=uint8


In [22]:
send_frame(unpacked_frame)

Frame sent successfully.


In [21]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
#fname=f"{timestamp}-image-output"
fname=f"image-output"
save_img(fname=fname, tensor=unpacked_frame)

'image-output.png'

# playground

In [ ]:
hasattr(ol.axi_vdma_0, 'write')  # should return True


In [ ]:
img_to_axis(ol.img2axis_0,buff_o,True,88)

In [ ]:
help(VideoMode)

In [ ]:
dir(ol.axi_vdma_0)

In [ ]:
ol.axi_vdma_0.framecount

In [ ]:
len(ol.axi_vdma_0.readchannel._frames)